In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

# =========================================================
# 1. USER SETTINGS
# =========================================================
BASE_DIR = Path(".")
PANEL_PATH = BASE_DIR / "Master_Modeling_Panel_v2.csv"

SUMMARY_CSV = BASE_DIR / "Feature_Block_Ablations_v1_Summary.csv"
CV_FOLD_METRICS_CSV = BASE_DIR / "Feature_Block_Ablations_v1_DevCV_Fold_Metrics.csv"
PREDICTIONS_CSV = BASE_DIR / "Feature_Block_Ablations_v1_Predictions.csv"
SPEC_SUMMARY_CSV = BASE_DIR / "Feature_Block_Ablations_v1_Spec_Summary.csv"

# Frozen winning pipeline from updated N3 (overall winner so far)
XGB_PARAMS = {
    "n_estimators": 800,
    "max_depth": 3,
    "learning_rate": 0.01,
    "min_child_weight": 10,
    "subsample": 0.6,
    "colsample_bytree": 0.5,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": 0,
}

TARGET_COL = "targeted_in_year"
CATEGORICAL_FEATURE = "ff17_for_model"
ID_COLS = ["permno", "year"]

# =========================================================
# 2. LOAD PANEL
# =========================================================
df = pd.read_csv(PANEL_PATH, low_memory=False)
df = df[df["year"].between(2010, 2024)].copy()
df[CATEGORICAL_FEATURE] = df["ff17_short"].fillna("Unclassified").astype(str)

# =========================================================
# 3. LOCKED RAW PREDICTOR SET
# =========================================================
# Core financial baseline blocks
accounting_continuous = [
    "roe",
    "assets_to_equity",
    "current_ratio",
    "ebitda",
    "ebitda_margin",
    "sales_to_assets",
    "sales_growth",
    "interest_coverage",
    "net_debt_to_ebitda",
    "fcf_to_capex",
]

market_continuous = [
    "market_cap",
    "ret_3m",
    "ret_6m",
    "ret_1y",
    "ret_2y",
    "ret_3y",
    "ret_5y",
    "volatility_30d",
    "volatility_90d",
    "volatility_180d",
    "turnover_30d",
]

valuation_payout_continuous = [
    "dividend_payout_ratio",
    "buyback_yield",
    "price_to_book",
    "ev_to_sales",
    "ev_to_ebitda",
    "tobins_q",
    "fcf_yield",
]

history_continuous = [
    "prior_campaign_count_3y",
    "prior_settlement_count_3y",
    "prior_management_favorable_count_3y",
    "prior_dissident_favorable_count_3y",
    "prior_campaign_count_5y",
    "prior_settlement_count_5y",
    "prior_management_favorable_count_5y",
    "prior_dissident_favorable_count_5y",
]

history_binary = [
    "prior_activism_3y",
    "prior_activism_5y",
    "history_3y_observed",
    "history_5y_observed",
]

# Enrichment blocks
relative_continuous = [
    "ff49_other_permno_years_in_category",  # relative-feature quality indicator
    "ret_3m_rel_peer",
    "ret_6m_rel_peer",
    "ret_1y_rel_peer",
    "ret_2y_rel_peer",
    "ret_3y_rel_peer",
    "ret_5y_rel_peer",
    "log_ev_to_sales_rel_peer",
    "log_price_to_book_rel_peer",
    "log_tobins_q_rel_peer",
    "log_ev_to_ebitda_rel_peer",
    "ebitda_margin_rel_peer",
    "sales_growth_rel_peer",
    "roe_rel_peer",
]

relative_binary = [
    "ff49_unclassified",  # relative-feature quality indicator
]

governance_continuous = [
    "board_size",
    "board_female_ratio",
    "ceo_tenure",
]

governance_binary = [
    # ISS governance provisions
    "classified_board",
    "poison_pill",
    "dual_class",
    "iss_match_found",
    "rm_stale_gt_730",
    # BoardEx board helpers
    "board_match_found",
    "board_stale_gt_365",
    # BoardEx CEO features/helpers
    "ceo_is_female",
    "ceo_chair_duality",
    "ceo_match_found",
    "ceo_stale_gt_365",
]

ownership_continuous = [
    "top10_inst_conc_within_13f",
    "inst_ownership_pct_13f",
]

ownership_binary = [
    "inst_match_found",
]

full_continuous = (
    accounting_continuous
    + market_continuous
    + valuation_payout_continuous
    + history_continuous
    + relative_continuous
    + governance_continuous
    + ownership_continuous
)
full_binary = (
    history_binary
    + relative_binary
    + governance_binary
    + ownership_binary
)
full_predictors = full_continuous + full_binary + [CATEGORICAL_FEATURE]

required_cols = full_predictors + [TARGET_COL] + ID_COLS
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

# =========================================================
# 4. ABLATION DEFINITIONS (v4 plan)
#    IMPORTANT: block drops also remove associated helper/coverage indicators.
# =========================================================
ABLATIONS = {
    "A1": {
        "description": "Baseline feature set only (accounting + market + valuation/payout + activism history)",
        "continuous": accounting_continuous + market_continuous + valuation_payout_continuous + history_continuous,
        "binary": history_binary,
    },
    "A2": {
        "description": "Full panel minus peer-relative block",
        "continuous": [c for c in full_continuous if c not in set(relative_continuous)],
        "binary": [b for b in full_binary if b not in set(relative_binary)],
    },
    "A3": {
        "description": "Full panel minus governance block (ISS + BoardEx combined)",
        "continuous": [c for c in full_continuous if c not in set(governance_continuous)],
        "binary": [b for b in full_binary if b not in set(governance_binary)],
    },
    "A4": {
        "description": "Full panel minus ownership block (13F + ownership helper)",
        "continuous": [c for c in full_continuous if c not in set(ownership_continuous)],
        "binary": [b for b in full_binary if b not in set(ownership_binary)],
    },
    "A5": {
        "description": "Full panel reference specification",
        "continuous": full_continuous.copy(),
        "binary": full_binary.copy(),
    },
}

# =========================================================
# 5. METRICS / PIPELINE HELPERS
# =========================================================
def evaluate_predictions(y_true: pd.Series, y_prob: np.ndarray) -> dict:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    return {
        "pr_auc": average_precision_score(y_true, y_prob),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "brier_score": brier_score_loss(y_true, y_prob),
    }


FF17_CATEGORIES = [
    "Cars",
    "Chems",
    "Clths",
    "Cnstr",
    "Cnsum",
    "Durbl",
    "FabPr",
    "Finan",
    "Food",
    "Machn",
    "Mines",
    "Oil",
    "Other",  # omitted reference
    "Rtail",
    "Steel",
    "Trans",
    "Utils",
    "Unclassified",
]


def build_pipeline(continuous_features: list[str], binary_features: list[str]) -> Pipeline:
    continuous_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
        ]
    )

    binary_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value="Unclassified")),
            (
                "onehot",
                OneHotEncoder(
                    categories=[FF17_CATEGORIES],
                    drop=["Other"],
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
            ),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("cont", continuous_transformer, continuous_features),
            ("bin", binary_transformer, binary_features),
            ("ff17", categorical_transformer, [CATEGORICAL_FEATURE]),
        ],
        remainder="drop",
    )

    model = XGBClassifier(**XGB_PARAMS)

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )


CV_FOLD_DEFS = [
    {"fold": "fold_1", "train_years": list(range(2010, 2016)), "val_years": [2016]},
    {"fold": "fold_2", "train_years": list(range(2010, 2017)), "val_years": [2017]},
    {"fold": "fold_3", "train_years": list(range(2010, 2018)), "val_years": [2018]},
    {"fold": "fold_4", "train_years": list(range(2010, 2019)), "val_years": [2019]},
    {"fold": "fold_5", "train_years": list(range(2010, 2020)), "val_years": [2020]},
    {"fold": "fold_6", "train_years": list(range(2010, 2021)), "val_years": [2021]},
]

# =========================================================
# 6. RUN ABLATIONS
# =========================================================
all_fold_rows: list[dict] = []
all_prediction_frames: list[pd.DataFrame] = []
summary_rows: list[dict] = []
spec_rows: list[dict] = []

run_start = time.time()
print("Starting feature-block ablation stage...")
print(f"Ablations to run: {', '.join(ABLATIONS.keys())}")
print()

for ablation_id, ablation_cfg in ABLATIONS.items():
    ablation_start = time.time()
    continuous_features = ablation_cfg["continuous"]
    binary_features = ablation_cfg["binary"]
    predictor_list = continuous_features + binary_features + [CATEGORICAL_FEATURE]

    dropped_features = [c for c in full_predictors if c not in predictor_list]

    spec_rows.append(
        {
            "ablation_id": ablation_id,
            "description": ablation_cfg["description"],
            "continuous_predictor_count": len(continuous_features),
            "binary_predictor_count": len(binary_features),
            "categorical_predictor_count": 1,
            "total_predictor_count": len(predictor_list),
            "dropped_predictor_count": len(dropped_features),
            "included_predictors": " | ".join(predictor_list),
            "dropped_predictors": " | ".join(dropped_features),
            "uses_frozen_winner_pipeline": True,
            "frozen_model_family": "XGBoost",
            "frozen_model_variant": "updated_N3_no_winsor",
            "imbalance_choice": "none",
        }
    )

    print(f"[{ablation_id}] {ablation_cfg['description']}")
    print(f"[{ablation_id}] predictors used: {len(predictor_list)} | dropped: {len(dropped_features)}")

    fold_rows = []
    prediction_frames = []

    for i, fold_def in enumerate(CV_FOLD_DEFS, start=1):
        train_years = fold_def["train_years"]
        val_years = fold_def["val_years"]
        fold_name = fold_def["fold"]

        train_df = df[df["year"].isin(train_years)].copy()
        val_df = df[df["year"].isin(val_years)].copy()

        X_train = train_df[predictor_list].copy()
        y_train = train_df[TARGET_COL].copy()
        X_val = val_df[predictor_list].copy()
        y_val = val_df[TARGET_COL].copy()

        print(
            f"[{ablation_id}] [CV] Fold {i}/6 | train={min(train_years)}-{max(train_years)} | "
            f"val={min(val_years)} | train_rows={len(train_df)} | val_rows={len(val_df)}"
        )

        pipeline = build_pipeline(continuous_features, binary_features)
        pipeline.fit(X_train, y_train)
        val_prob = pipeline.predict_proba(X_val)[:, 1]

        fold_metrics = evaluate_predictions(y_val, val_prob)
        fold_metrics.update(
            {
                "ablation_id": ablation_id,
                "description": ablation_cfg["description"],
                "fold": fold_name,
                "train_year_start": min(train_years),
                "train_year_end": max(train_years),
                "val_year_start": min(val_years),
                "val_year_end": max(val_years),
                "train_rows": len(train_df),
                "train_positives": int(y_train.sum()),
                "train_positive_rate": float(y_train.mean()),
                "val_rows": len(val_df),
                "val_positives": int(y_val.sum()),
                "val_positive_rate": float(y_val.mean()),
                "mean_predicted_probability": float(np.mean(val_prob)),
                "predictor_count": len(predictor_list),
            }
        )
        fold_rows.append(fold_metrics)

        prediction_frames.append(
            pd.DataFrame(
                {
                    "ablation_id": ablation_id,
                    "description": ablation_cfg["description"],
                    "permno": val_df["permno"].to_numpy(),
                    "year": val_df["year"].to_numpy(),
                    "evaluation_stage": "development_cv",
                    "fold": fold_name,
                    "actual": y_val.to_numpy(),
                    "predicted_probability": val_prob,
                }
            )
        )

    fold_df = pd.DataFrame(fold_rows)
    all_fold_rows.extend(fold_rows)

    # Refit on full 2010-2021, evaluate on 2022-2024
    dev_df = df[df["year"].between(2010, 2021)].copy()
    test_df = df[df["year"].between(2022, 2024)].copy()

    X_dev = dev_df[predictor_list].copy()
    y_dev = dev_df[TARGET_COL].copy()
    X_test = test_df[predictor_list].copy()
    y_test = test_df[TARGET_COL].copy()

    print(
        f"[{ablation_id}] [TEST] Refit on 2010-2021 | dev_rows={len(dev_df)} | test_rows={len(test_df)}"
    )

    final_pipeline = build_pipeline(continuous_features, binary_features)
    final_pipeline.fit(X_dev, y_dev)
    test_prob = final_pipeline.predict_proba(X_test)[:, 1]
    test_metrics = evaluate_predictions(y_test, test_prob)

    prediction_frames.append(
        pd.DataFrame(
            {
                "ablation_id": ablation_id,
                "description": ablation_cfg["description"],
                "permno": test_df["permno"].to_numpy(),
                "year": test_df["year"].to_numpy(),
                "evaluation_stage": "final_test",
                "fold": "final_test",
                "actual": y_test.to_numpy(),
                "predicted_probability": test_prob,
            }
        )
    )
    all_prediction_frames.extend(prediction_frames)

    summary_rows.append(
        {
            "ablation_id": ablation_id,
            "description": ablation_cfg["description"],
            "predictor_count": len(predictor_list),
            "dropped_predictor_count": len(dropped_features),
            "cv_mean_pr_auc": float(fold_df["pr_auc"].mean()),
            "cv_std_pr_auc": float(fold_df["pr_auc"].std(ddof=1)),
            "cv_mean_roc_auc": float(fold_df["roc_auc"].mean()),
            "cv_std_roc_auc": float(fold_df["roc_auc"].std(ddof=1)),
            "cv_mean_brier_score": float(fold_df["brier_score"].mean()),
            "cv_std_brier_score": float(fold_df["brier_score"].std(ddof=1)),
            "final_test_pr_auc": float(test_metrics["pr_auc"]),
            "final_test_roc_auc": float(test_metrics["roc_auc"]),
            "final_test_brier_score": float(test_metrics["brier_score"]),
            "final_test_rows": int(len(test_df)),
            "final_test_positives": int(y_test.sum()),
            "final_test_positive_rate": float(y_test.mean()),
            "final_test_mean_predicted_probability": float(np.mean(test_prob)),
            "runtime_minutes": (time.time() - ablation_start) / 60.0,
        }
    )

    print(
        f"[{ablation_id}] complete | cv_mean_pr_auc={fold_df['pr_auc'].mean():.4f} | "
        f"final_test_pr_auc={test_metrics['pr_auc']:.4f} | runtime={(time.time() - ablation_start)/60.0:.2f} min"
    )
    print()

# =========================================================
# 7. SAVE OUTPUTS
# =========================================================
summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values(by="ablation_id").reset_index(drop=True)

# Delta vs A5 full reference
ref_row = summary_df.loc[summary_df["ablation_id"] == "A5"].iloc[0]
summary_df["delta_vs_A5_cv_mean_pr_auc"] = summary_df["cv_mean_pr_auc"] - float(ref_row["cv_mean_pr_auc"])
summary_df["delta_vs_A5_final_test_pr_auc"] = summary_df["final_test_pr_auc"] - float(ref_row["final_test_pr_auc"])
summary_df["delta_vs_A5_final_test_roc_auc"] = summary_df["final_test_roc_auc"] - float(ref_row["final_test_roc_auc"])
summary_df["delta_vs_A5_final_test_brier_score"] = summary_df["final_test_brier_score"] - float(ref_row["final_test_brier_score"])

summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_fold_rows).to_csv(CV_FOLD_METRICS_CSV, index=False)
pd.concat(all_prediction_frames, ignore_index=True).to_csv(PREDICTIONS_CSV, index=False)
pd.DataFrame(spec_rows).to_csv(SPEC_SUMMARY_CSV, index=False)

print("Feature-block ablation stage complete.")
print(f"Total runtime: {(time.time() - run_start)/60.0:.2f} minutes")
print(f"Saved summary to: {SUMMARY_CSV}")
print(f"Saved CV fold metrics to: {CV_FOLD_METRICS_CSV}")
print(f"Saved predictions to: {PREDICTIONS_CSV}")
print(f"Saved spec summary to: {SPEC_SUMMARY_CSV}")

Starting feature-block ablation stage...
Ablations to run: A1, A2, A3, A4, A5

[A1] Baseline feature set only (accounting + market + valuation/payout + activism history)
[A1] predictors used: 41 | dropped: 32
[A1] [CV] Fold 1/6 | train=2010-2015 | val=2016 | train_rows=24053 | val_rows=3888
[A1] [CV] Fold 2/6 | train=2010-2016 | val=2017 | train_rows=27941 | val_rows=3811
[A1] [CV] Fold 3/6 | train=2010-2017 | val=2018 | train_rows=31752 | val_rows=3794
[A1] [CV] Fold 4/6 | train=2010-2018 | val=2019 | train_rows=35546 | val_rows=3805
[A1] [CV] Fold 5/6 | train=2010-2019 | val=2020 | train_rows=39351 | val_rows=3907
[A1] [CV] Fold 6/6 | train=2010-2020 | val=2021 | train_rows=43258 | val_rows=4535
[A1] [TEST] Refit on 2010-2021 | dev_rows=47793 | test_rows=13147
[A1] complete | cv_mean_pr_auc=0.1540 | final_test_pr_auc=0.2791 | runtime=0.24 min

[A2] Full panel minus peer-relative block
[A2] predictors used: 58 | dropped: 15
[A2] [CV] Fold 1/6 | train=2010-2015 | val=2016 | train_rows=